In [ ]:
from gsloc.inference.test import TestConfig, Test
from pathlib import Path
from gsloc.models import opr_graph_extention as network 
import torch
from torchvision.transforms import functional as F
from mmpr.models import MegaLoc
from gsloc.datasets import Replica

from torchvision import transforms as T
from gsloc.utils.visual import plot_metrics_from_parquet, plot_metrics_from_experiment_dir

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


2026-05-07 01:50:12.039 | WARNING  | opr.optional_deps:warn_once:115 - MinkowskiEngine is not available. sparse convolutions will be disabled. See the documentation for installation instructions


In [2]:
weights_path = Path("/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/best_model.pth")
ckpt = torch.load(weights_path, map_location="cpu", weights_only=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OPR_GAT_graph_encoder = network.OPR_GATGraphEncoder(
    in_dim=4,
    hidden_dim=512,
    n_layers=1,
    num_node_classes=529, 
    node_emb_dim=128,
    num_edge_classes=41,
    edge_emb_dim=128,
    proj_dim=256,
    edge_cont_dim=10,
    dropout=0.1,
    heads=4
    ).to(device)
    
# megaloc = torch.hub.load("gmberton/MegaLoc", "get_trained_model")
# image_encoder = megaloc.to(device)

graph_model1 = network.OPR_MultiModalVPRGraphEncoder(
    graph_encoder=OPR_GAT_graph_encoder,
    image_encoder=None,
    image_out_dim=8448,
    graph_out_dim=256,
    fusion_dim=8448,
    normalize=True,
    graph_fusion_scale=0.05,
    freeze_image_encoder=True,
    mode="graph")

missing, unexpected = graph_model1.load_state_dict(ckpt["model_state_dict"], strict=False)
ignored_unexpected_prefixes = ("image_encoder.", "graph_encoder.convs.")
unexpected_other = [k for k in unexpected if not k.startswith(ignored_unexpected_prefixes)]
if unexpected_other:
    raise RuntimeError(f"Unexpected checkpoint keys: {unexpected_other}")
# ``missing`` includes MegaLoc hub weights and GINE conv params; those ckpt tensors appear under ``ignored_unexpected_prefixes``.

graph_model1.to(device)
graph_model1.eval()

KeyboardInterrupt: 

In [ ]:
megaLoc = MegaLoc()
megaLoc.to(device)
megaLoc.eval()

In [ ]:
dataset_path = Path("/mnt/external_usb_hdd/6YL/Datasets/Replica")
test_dir = Path("/home/kartashov_ga/projects/GSLoc/data/tests/26-05-07/Replica/makarov/256xMegaloc")
index_path = "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-07/Replica/makarov/256_graph_index"
rerank_index_path = "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-07/Replica/megaloc_index"
query_cache_path = "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-07/Replica/makarov/256_graph_query_cache"
rerank_query_cache_path = "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-07/Replica/megaloc_query_cache"

similarity_names = ["room-sim", "pose-far-sim", "pose-near-sim"]
filter_names = ["base_seq_report", "nearfilter_seq_report", "farfilter_seq_report"]

graph_path = "SceneGraphs_Makarov_TEST_pt"
edge_normalizer_path="/mnt/external_usb_hdd/6YL/Datasets/3RScan/weights/graphs/gatv1/edge_normalizer (1).pt"
scene_list_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/test_resplit_scans.txt"
room_json_path = "/mnt/external_usb_hdd/6YL/Datasets/3RScan/files/3RScan.json"

bench_report_dir = test_dir / similarity_names[1]
frames_path = "/home/kartashov_ga/projects/GSLoc/data/tests/26-05-06/GT/256_pure_graph/frames.npz"

seq_filter_kwargs_list = [{
    "seq_similarity_filter_mode": "none",
},
{
    "seq_similarity_filter_mode": "pose",
    "seq_similarity_trans_tol_m": 0.5,
    "seq_similarity_rot_tol_deg": 15
},{
    "seq_similarity_filter_mode": "pose",
    "seq_similarity_trans_tol_m": 1,
    "seq_similarity_rot_tol_deg": 30
},
]
models = [graph_model1]
rerank_models = [megaLoc]

graph_path_list = [graph_path, "SceneGraphs_real_classes_pt_compact"]
similarity_kwargs_list = [
    {
        "mode": "room",
    },
    {
        "mode": "pose",
        "trans_tol_m": 3,
        "rot_tol_deg": 180
    },
    {
        "mode": "pose",
        "trans_tol_m": 2,
        "rot_tol_deg": 90
    },
]


image_transform_fn = T.Compose([
    T.ToTensor(),
    T.Lambda(lambda x: F.rotate(x, angle=-90)),  # 90° clockwise
    T.Normalize(mean=[0.44420420130352495, 0.41322746532289134, 0.3678658064565412], std=[0.24352604373543688, 0.24045797651069503, 0.24250136992133814]),
    T.Resize([322, 322], antialias=True)
])

cfg = TestConfig(
    dataset_path=dataset_path,
    test_path=test_dir,
    index_path=index_path,
    rerank_index_path=rerank_index_path,
    query_cache_path=query_cache_path,
    rerank_query_cache_path=rerank_query_cache_path,
    bench_report_path=bench_report_dir,
    graph_path=graph_path,   
    dataset_class=ThreeRScan,
    filter_kwargs={"similarity_filter_mode": "none", "similarity_trans_tol_m": 2, "similarity_rot_tol_deg": 90},
    seq_filter_kwargs=seq_filter_kwargs_list[1],
    scene_list_path=scene_list_path,
    room_json_path=room_json_path,
    edge_normalizer_path=edge_normalizer_path,
    image_transform_fn=image_transform_fn,
    graph_feat_dim=4,
    graph_edge_attr_dim=10,
    graph_rotate=True,
    device=device,
    batch_size=16,
    num_workers=4,
    model=graph_model1,
    # rerank_model=megaLoc,
    rerank_k=500,
    per_frame_k_used=25,
    final_k=25,
    seq_lengths=[1, 2, 3, 5, 7, 10, 15, 20, 25, 30, 35],
    recall_at_k=[1, 5, 10, 25],
    similarity_kwargs=similarity_kwargs_list[0],
    std_mode="global",
    scene_df_field="scene",
    pose_df_field="pose",
    frames_path=frames_path
)

In [ ]:
test = Test(cfg)
test.run()

In [ ]:
cfg.rerank_model = megaLoc
for i, similarity_kwargs in enumerate(similarity_kwargs_list):
    cfg.similarity_kwargs = similarity_kwargs
    cfg.bench_report_path = cfg.test_path  / similarity_names[i]
    cfg.frames_path = cfg.test_path / "frames.npz"
    test = Test(cfg)
    test.run()